# BÀI THỰC HÀNH TUẦN 2: THUẬT TOÁN NAÏVE BAYES
- **Họ và tên**: Nguyễn Hải Anh  
- **MSSV**: 24001968  
- **Môn học**: Machine Learning  

---

## 1. Lý thuyết: Tại sao dùng hàm mật độ xác suất (PDF) & Công thức Bayes hỗn hợp?

- **Vấn đề**: Với biến liên tục $X \in \mathbb{R}$, xác suất tại một điểm chính xác luôn bằng 0 ($P(X = x) = 0$). Nếu áp dụng công thức Bayes thông thường sẽ rơi vào dạng vô định $\frac{0}{0}$.
- **Cách giải quyết**: Xét $X$ trong một khoảng rất nhỏ quanh $x$: $[x - \epsilon, x + \epsilon]$ với $\epsilon \to 0$. Khi đó:
  - $P(x - \epsilon \le X \le x + \epsilon \mid Y = k) \approx 2\epsilon \cdot f_{X \mid Y = k}(x)$
  - $P(x - \epsilon \le X \le x + \epsilon) \approx 2\epsilon \cdot f_X(x)$
- **Công thức Bayes hỗn hợp (Hybrid Bayes Rule)**: Lấy tỷ số Bayes, thừa số $2\epsilon$ ở cả tử và mẫu bị triệt tiêu:
  $$P(Y = k \mid X = x) = \frac{P(Y = k) \cdot f_{X \mid Y = k}(x)}{f_X(x)}$$
  Trong đó $f_{X \mid Y = k}(x)$ là hàm mật độ xác suất của phân phối chuẩn Gaussian (PDF):
  $$f(x; \mu, \sigma) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x - \mu)^2}{2\sigma^2}\right)$$
- **Quy tắc dự đoán**: Vì mẫu số $f_X(x)$ giống nhau cho mọi lớp, ta chỉ cần so sánh tử số:
  $$\hat{y} = \arg\max_k \left[ P(Y = k) \cdot f_{X \mid Y = k}(x) \right]$$

## 2. Bài tập 1: Phân loại hoa Iris với 4 đặc trưng (Gaussian NB)
- Sử dụng đủ 4 đặc trưng: sepal_length, sepal_width, petal_length, petal_width.
- Tỉ lệ Train : Test = 4 : 1 (80% Train, 20% Test).
- So sánh cách tự cài đặt (Scratch) và thư viện Scikit-learn.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Đọc dữ liệu Iris
df_iris = pd.read_csv('iris.csv')
feature_cols = [c for c in df_iris.columns if c != 'variety']
X = df_iris[feature_cols].values
y_raw = df_iris['variety'].values

# Đổi nhãn sang số: 0, 1, 2
classes, y = np.unique(y_raw, return_inverse=True)

# 2. Chia tập Train/Test theo tỉ lệ 4:1 (80% : 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Số mẫu Train: {len(X_train)}, Số mẫu Test: {len(X_test)}")

# 3. Tự cài đặt Gaussian Naive Bayes (From Scratch)
means = np.array([X_train[y_train == c].mean(axis=0) for c in range(len(classes))])
stds = np.array([X_train[y_train == c].std(axis=0, ddof=0) for c in range(len(classes))])
priors = np.array([(y_train == c).mean() for c in range(len(classes))])

def predict_scratch(X_val):
    preds = []
    for x in X_val:
        # Tính log-posterior để tránh tràn số dưới
        scores = [np.log(priors[c]) + np.sum(norm.logpdf(x, loc=means[c], scale=stds[c])) for c in range(len(classes))]
        preds.append(np.argmax(scores))
    return np.array(preds)

y_pred_scratch = predict_scratch(X_test)

# 4. Dùng thư viện Scikit-learn
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred_sk = gnb.predict(X_test)

print("Độ chính xác Scratch:   ", accuracy_score(y_test, y_pred_scratch))
print("Độ chính xác Scikit-learn:", accuracy_score(y_test, y_pred_sk))
print("Hai cách cho kết quả giống nhau 100%:", np.all(y_pred_scratch == y_pred_sk))
print("\nBáo cáo phân loại chi tiết:\n", classification_report(y_test, y_pred_sk, target_names=classes))


Số mẫu Train: 120, Số mẫu Test: 30
Độ chính xác Scratch:    0.9666666666666667
Độ chính xác Scikit-learn: 0.9666666666666667
Hai cách cho kết quả giống nhau 100%: True

Báo cáo phân loại chi tiết:
               precision    recall  f1-score   support

      Setosa       1.00      1.00      1.00        10
  Versicolor       1.00      0.90      0.95        10
   Virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



## 3. Bài tập 2: Phân loại ung thư vú Wisconsin (Gaussian NB)
- Đọc dữ liệu `breast-cancer-wisconsin.data` bằng Pandas (theo mẫu ở cuối tệp nguồn `Gaussian_NB.ipynb`).
- Điền 16 giá trị thiếu `?` ở cột `bare_nuclei` bằng trung vị (`median`) theo từng lớp bệnh.
- Tách 80 mẫu lành tính (2) và 40 mẫu ác tính (4) làm Test (tổng 120 mẫu), 579 mẫu còn lại làm Train.
- Đánh giá mô hình bằng Accuracy, Precision, Recall, F1.

In [2]:
col_names = ['id', 'clump_thickness', 'cell_size_uniformity', 'cell_shape_uniformity',
             'marginal_adhesion', 'single_epithelial_size', 'bare_nuclei',
             'bland_chromatin', 'normal_nucleoli', 'mitoses', 'class']
df_cancer = pd.read_csv('breast-cancer-wisconsin.data', header=None, names=col_names)

# Xử lý giá trị thiếu '?' bằng trung vị (median) theo lớp
df_cancer['bare_nuclei'] = pd.to_numeric(df_cancer['bare_nuclei'].replace('?', np.nan))
df_cancer['bare_nuclei'] = df_cancer.groupby('class')['bare_nuclei'].transform(lambda s: s.fillna(s.median()))

# Tách 80 mẫu lành tính (2) và 40 mẫu ác tính (4) làm Test
test_benign = df_cancer[df_cancer['class'] == 2].sample(80, random_state=42)
test_malignant = df_cancer[df_cancer['class'] == 4].sample(40, random_state=42)
test_df = pd.concat([test_benign, test_malignant])
train_df = df_cancer.drop(test_df.index)

print(f"Tập Train: {len(train_df)} mẫu, Tập Test: {len(test_df)} mẫu (80 lành, 40 ác)")

feat_cols = [c for c in col_names if c not in ['id', 'class']]
X_train_c = train_df[feat_cols].values
y_train_c = train_df['class'].values
X_test_c = test_df[feat_cols].values
y_test_c = test_df['class'].values

# Huấn luyện GaussianNB
model_cancer = GaussianNB()
model_cancer.fit(X_train_c, y_train_c)
y_pred_c = model_cancer.predict(X_test_c)

print(f"Accuracy:  {accuracy_score(y_test_c, y_pred_c)*100:.2f}%")
print(f"Recall ca ác tính (rất quan trọng trong y tế): {classification_report(y_test_c, y_pred_c, output_dict=True)['4']['recall']*100:.2f}%")
print("\nBáo cáo phân loại:\n", classification_report(y_test_c, y_pred_c, target_names=['Lành tính (2)', 'Ác tính (4)']))


Tập Train: 579 mẫu, Tập Test: 120 mẫu (80 lành, 40 ác)
Accuracy:  96.67%
Recall ca ác tính (rất quan trọng trong y tế): 95.00%

Báo cáo phân loại:
                precision    recall  f1-score   support

Lành tính (2)       0.97      0.97      0.97        80
  Ác tính (4)       0.95      0.95      0.95        40

     accuracy                           0.97       120
    macro avg       0.96      0.96      0.96       120
 weighted avg       0.97      0.97      0.97       120



## 4. Bài tập 3: Phân loại văn bản ẩm thực Bắc (B) - Nam (N)
- Tập Train gồm 4 văn bản D1, D2, D3 (lớp B) và D4 (lớp N).
- Cần dự đoán 2 văn bản Test D5, D6 thuộc lớp Bắc (B) hay Nam (N).
- Thử nghiệm với cả MultinomialNB (tần suất từ) và BernoulliNB (nhị phân có/không).

In [3]:
from sklearn.naive_bayes import MultinomialNB, BernoulliNB

# Từ điển 9 từ: Hà-Nội, phở, cháo-lòng, bún-chả, ô-mai, bánh-giò, Sài-gòn, hủ-tiếu, bánh-bò
# 1. Dạng đếm số lần xuất hiện từ (Multinomial)
X_train_mnb = np.array([
    [2, 1, 1, 0, 0, 0, 0, 0, 0], # D1: B
    [1, 1, 0, 1, 1, 0, 0, 0, 0], # D2: B
    [0, 1, 0, 0, 1, 1, 0, 0, 0], # D3: B
    [0, 1, 0, 0, 0, 0, 1, 1, 1]  # D4: N
])
y_train_text = np.array(['B', 'B', 'B', 'N'])

X_test_mnb = np.array([
    [2, 0, 0, 1, 0, 0, 0, 1, 0], # D5
    [0, 1, 0, 0, 0, 0, 0, 1, 1]  # D6
])

clf_mnb = MultinomialNB(alpha=1.0)
clf_mnb.fit(X_train_mnb, y_train_text)
print("MultinomialNB dự đoán:")
print(" - D5 thuộc lớp:", clf_mnb.predict(X_test_mnb)[0])
print(" - D6 thuộc lớp:", clf_mnb.predict(X_test_mnb)[1])
print("Xác suất dự đoán [B, N]:\n", clf_mnb.predict_proba(X_test_mnb))

# 2. Dạng nhị phân có/không (Bernoulli)
X_train_bnb = (X_train_mnb > 0).astype(int)
X_test_bnb = (X_test_mnb > 0).astype(int)

clf_bnb = BernoulliNB(alpha=1.0)
clf_bnb.fit(X_train_bnb, y_train_text)
print("\nBernoulliNB dự đoán:")
print(" - D5 thuộc lớp:", clf_bnb.predict(X_test_bnb)[0])
print(" - D6 thuộc lớp:", clf_bnb.predict(X_test_bnb)[1])
print("Xác suất dự đoán [B, N]:\n", clf_bnb.predict_proba(X_test_bnb))


MultinomialNB dự đoán:
 - D5 thuộc lớp: B
 - D6 thuộc lớp: N
Xác suất dự đoán [B, N]:
 [[0.89548823 0.10451177]
 [0.29175335 0.70824665]]

BernoulliNB dự đoán:
 - D5 thuộc lớp: B
 - D6 thuộc lớp: N
Xác suất dự đoán [B, N]:
 [[0.76554295 0.23445705]
 [0.16948581 0.83051419]]


## 5. Bài tập 4: Phân loại Email Spam với tập Ling-Spam (`ex6DataPrepared`)
- Bộ dữ liệu `ex6DataPrepared` gồm 960 email tiếng Anh (700 Train, 260 Test), từ điển 2500 từ vựng.
- Đọc dữ liệu ma trận thưa bằng `scipy.sparse.coo_matrix`.
- So sánh MultinomialNB và BernoulliNB trên tập Test.

In [4]:
from scipy.sparse import coo_matrix

path_spam = 'ex6DataPrepared/'
nwords = 2500

def read_spam(data_fn, label_fn):
    with open(path_spam + label_fn, 'r') as f:
        labels = [int(x.strip()) for x in f if x.strip()]
    with open(path_spam + data_fn, 'r') as f:
        lines = [x.strip() for x in f if x.strip()]
    dat = np.zeros((len(lines), 3), dtype=int)
    for i, line in enumerate(lines):
        parts = line.split()
        dat[i, :] = [int(parts[0]), int(parts[1]), int(parts[2])]
    matrix = coo_matrix((dat[:, 2], (dat[:, 0] - 1, dat[:, 1] - 1)), shape=(len(labels), nwords))
    return matrix, np.array(labels)

train_data, train_label = read_spam('train-features.txt', 'train-labels.txt')
test_data, test_label = read_spam('test-features.txt', 'test-labels.txt')

# MultinomialNB
mnb_spam = MultinomialNB(alpha=1.0).fit(train_data, train_label)
acc_mnb = accuracy_score(test_label, mnb_spam.predict(test_data))

# BernoulliNB
bnb_spam = BernoulliNB(alpha=1.0).fit((train_data > 0).astype(int), train_label)
acc_bnb = accuracy_score(test_label, bnb_spam.predict((test_data > 0).astype(int)))

print(f"Độ chính xác MultinomialNB (Train 700): {acc_mnb*100:.2f}%")
print(f"Độ chính xác BernoulliNB   (Train 700): {acc_bnb*100:.2f}%")

# Khảo sát khi tập Train ít mẫu hơn
print("\nKhảo sát MultinomialNB với các cỡ mẫu Train khác nhau:")
for size in [50, 100, 400]:
    d_sub, l_sub = read_spam(f'train-features-{size}.txt', f'train-labels-{size}.txt')
    m = MultinomialNB(alpha=1.0).fit(d_sub, l_sub)
    print(f" - Train {size:3d} mẫu: Test Accuracy = {accuracy_score(test_label, m.predict(test_data))*100:.2f}%")


Độ chính xác MultinomialNB (Train 700): 98.08%
Độ chính xác BernoulliNB   (Train 700): 85.38%

Khảo sát MultinomialNB với các cỡ mẫu Train khác nhau:
 - Train  50 mẫu: Test Accuracy = 97.31%
 - Train 100 mẫu: Test Accuracy = 97.69%
 - Train 400 mẫu: Test Accuracy = 97.69%


## 6. Bài tập 5: Dự đoán loại ẩm thực đơn hàng (`multinomial_nb_orders.csv`)
- Dữ liệu gồm 3000 đơn hàng với 8 thuộc tính danh mục rời rạc, dự đoán 10 loại ẩm thực (`cuisine`).
- Tỉ lệ Train : Test = 7 : 3.
- Áp dụng One-Hot Encoding để chuyển sang dạng đếm rời rạc cho MultinomialNB.

In [5]:
from sklearn.preprocessing import OneHotEncoder

df_orders = pd.read_csv('multinomial_nb_orders (1).csv')
X_orders = df_orders.drop(columns=['cuisine'])
y_orders = df_orders['cuisine']

X_tr_o, X_te_o, y_tr_o, y_te_o = train_test_split(X_orders, y_orders, test_size=0.3, random_state=42, stratify=y_orders)

# One-hot encoding
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_tr_ohe = ohe.fit_transform(X_tr_o)
X_te_ohe = ohe.transform(X_te_o)

mnb_orders = MultinomialNB(alpha=1.0)
mnb_orders.fit(X_tr_ohe, y_tr_o)
y_pred_orders = mnb_orders.predict(X_te_ohe)

print(f"Độ chính xác MultinomialNB: {accuracy_score(y_te_o, y_pred_orders)*100:.2f}%")
print("Ghi chú: Bài toán 10 lớp, đoán ngẫu nhiên chỉ đạt 10%. Mô hình đạt trên 50% là rất tốt.")


Độ chính xác MultinomialNB: 50.33%
Ghi chú: Bài toán 10 lớp, đoán ngẫu nhiên chỉ đạt 10%. Mô hình đạt trên 50% là rất tốt.


## 7. Bài tập 6: Chẩn đoán 8 bệnh lý từ 30 triệu chứng lâm sàng (`bernoulli_nb_symptoms.csv`)
- Dữ liệu gồm 2500 bệnh nhân với 30 triệu chứng nhị phân (0 hoặc 1), chẩn đoán 8 loại bệnh lý.
- Tỉ lệ Train : Test = 6 : 4.
- Sử dụng BernoulliNB (mô hình chuẩn cho dữ liệu nhị phân 0/1).

In [6]:
df_symp = pd.read_csv('bernoulli_nb_symptoms.csv')
X_symp = df_symp.drop(columns=['condition'])
y_symp = df_symp['condition']

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_symp, y_symp, test_size=0.4, random_state=42, stratify=y_symp)

bnb_symp = BernoulliNB(alpha=1.0)
bnb_symp.fit(X_tr_s, y_tr_s)
y_pred_symp = bnb_symp.predict(X_te_s)

print(f"Độ chính xác BernoulliNB: {accuracy_score(y_te_s, y_pred_symp)*100:.2f}%")
print("Ghi chú: Bài toán 8 lớp, đoán ngẫu nhiên chỉ đạt 12.5%. Mô hình đạt ~50% (gấp 4 lần ngẫu nhiên).")


Độ chính xác BernoulliNB: 49.90%
Ghi chú: Bài toán 8 lớp, đoán ngẫu nhiên chỉ đạt 12.5%. Mô hình đạt ~50% (gấp 4 lần ngẫu nhiên).


## 8. Bảng tổng kết kết quả

| STT | Bài toán & Bộ dữ liệu | Mô hình Naive Bayes | Train / Test | Accuracy |
|:---:|:---|:---|:---:|:---:|
| **1** | Hoa Iris (4 đặc trưng) | GaussianNB (Scratch & Sklearn) | 120 / 30 | **96.67%** |
| **2** | Ung thư vú Wisconsin | GaussianNB (Xử lý missing value) | 579 / 120 | **96.67%** (Recall ác tính: 95.00%) |
| **3** | Văn bản ẩm thực Bắc - Nam | MultinomialNB & BernoulliNB | 4 / 2 | **100%** (D5 $\to$ B, D6 $\to$ N) |
| **4** | Email Spam (Ling-Spam `ex6`) | MultinomialNB / BernoulliNB | 700 / 260 | **98.08%** / **85.38%** |
| **5** | Đơn hàng trực tuyến (Orders) | MultinomialNB (One-Hot) | 2100 / 900 | **50.33%** (Gấp 5x ngẫu nhiên 10 lớp) |
| **6** | Chẩn đoán 8 bệnh lý (Symptoms) | BernoulliNB (30 triệu chứng) | 1500 / 1000 | **49.90%** (Gấp 4x ngẫu nhiên 8 lớp) |

**Kết luận ngắn gọn:**
- **GaussianNB**: Thích hợp cho biến liên tục tuân theo phân phối chuẩn.
- **MultinomialNB**: Tốt nhất cho dữ liệu đếm tần suất (phân loại văn bản/email spam đạt độ chính xác rất cao > 98%).
- **BernoulliNB**: Tối ưu cho dữ liệu nhị phân có/không (như triệu chứng lâm sàng).